In [ ]:
import sys
!{sys.executable} -m pip install -q mlxtend


: 

In [ ]:
!pip install -q mlxtend


In [ ]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
dataset_path = "employee_performance.csv"   # <-- kendi CSV path'in
MIN_SUPPORT = 0.05
MIN_CONFIDENCE = 0.60
MIN_LIFT = 1.0
MAX_LEN = 3  # itemset max uzunluğu (2-3 genelde yeter)
OUT_CSV = "apriori_rules.csv"
OUT_TEX = "apriori_rules_table.tex"
TOP_K = 10   # rapora koymak için en iyi kaç kural

In [ ]:

def pick_col(df, candidates):
    """CSV kolon adları space/snake_case farkı için: ilk bulunanı seçer."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Kolon bulunamadı. Denenenler: {candidates}\nMevcut kolonlar: {list(df.columns)[:30]} ...")


def qcut_ranked(series, q, labels):
    """qcut bazen duplicate edge verir (özellikle discrete datada). Rank ile stabilize eder."""
    r = series.rank(method="first")
    return pd.qcut(r, q=q, labels=labels)

In [ ]:
df = pd.read_csv(dataset_path)

In [ ]:
col_overtime = pick_col(df, ["Overtime Hours", "Overtime_Hours", "OvertimeHours"])
col_satisf   = pick_col(df, ["Employee Satisfaction Score", "Employee_Satisfaction_Score", "Satisfaction", "EmployeeSatisfactionScore"])
col_workhrs  = pick_col(df, ["Work Hours Per Week", "Work_Hours_Per_Week", "WorkHoursPerWeek"])
col_prom     = pick_col(df, ["Promotions", "Promotion", "Num_Promotions"])
col_perf     = pick_col(df, ["Performance Score", "Performance_Score", "PerformanceScore"])

use = df[[col_overtime, col_satisf, col_workhrs, col_prom, col_perf]].copy()

In [ ]:
use["Overtime_cat"] = qcut_ranked(use[col_overtime].astype(float), q=3, labels=["Low", "Medium", "High"])
use["Satisfaction_cat"] = qcut_ranked(use[col_satisf].astype(float), q=3, labels=["Low", "Medium", "High"])
use["WorkHours_cat"] = qcut_ranked(use[col_workhrs].astype(float), q=2, labels=["Low", "High"])

In [ ]:
use["Promotions_cat"] = np.where(use[col_prom].astype(float) > 0, "Yes", "No")

# Performance Score discrete olabilir (1-5). Eğer 1-5 ise cut daha mantıklı.
perf_vals = pd.to_numeric(use[col_perf], errors="coerce")
if perf_vals.dropna().nunique() <= 7:
    use["Performance_cat"] = pd.cut(perf_vals,
                                    bins=[-np.inf, 2, 3, np.inf],
                                    labels=["Low", "Medium", "High"],
                                    include_lowest=True)
else:
    use["Performance_cat"] = qcut_ranked(perf_vals.astype(float), q=3, labels=["Low", "Medium", "High"])

use = use.dropna()

In [ ]:
transactions = []
for _, row in use.iterrows():
    items = [
        f"Overtime={row['Overtime_cat']}",
        f"Satisfaction={row['Satisfaction_cat']}",
        f"WorkHours={row['WorkHours_cat']}",
        f"Promotions={row['Promotions_cat']}",
        f"Performance={row['Performance_cat']}",
    ]
    transactions.append(items)


In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
tx = pd.DataFrame(te_ary, columns=te.columns_)

# ---- Apriori ----
freq = apriori(tx, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)

In [ ]:
rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules[(rules["lift"] > MIN_LIFT)].copy()

# Pretty columns
def fs(itemset):
    return ", ".join(sorted(list(itemset)))

rules["antecedent_str"] = rules["antecedents"].apply(fs)
rules["consequent_str"] = rules["consequents"].apply(fs)

show = rules[["antecedent_str", "consequent_str", "support", "confidence", "lift"]] \
    .sort_values(["lift", "confidence", "support"], ascending=False) \
    .head(TOP_K)

In [ ]:
show.to_csv(OUT_CSV, index=False)

latex_df = show.copy()
latex_df["support"] = latex_df["support"].map(lambda x: f"{x:.2f}")
latex_df["confidence"] = latex_df["confidence"].map(lambda x: f"{x:.2f}")
latex_df["lift"] = latex_df["lift"].map(lambda x: f"{x:.2f}")

latex_table = latex_df.to_latex(index=False, escape=True,
                                caption="Selected association rules discovered using Apriori.",
                                label="tab:apriori_rules")

with open(OUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex_table)

print("Top rules:\n", show.to_string(index=False))
print(f"\nSaved: {OUT_CSV}")
print(f"Saved: {OUT_TEX}")